# Module 4: experimental data files, soil and plant nitrogen, N-limited production

This notebook follows the fourth day of a DSSAT training course, exercise by exercise:

1. **Creating crop measurement files**: the 1976 peanut shading study. Build its experiment file
   (`UFGA7609.PNX`, with an environmental modification that shades the crop), write its FileA and FileT from the
   course's spreadsheets (what the ATCreate tool does), and evaluate the simulation. Then the whole published
   study, and a sweep over when and how strongly the crop is shaded.
2. **Nitrogen-limited production**: the maize experiment `UFGA8213.MZX` with the water and nitrogen balances
   on, built as the exercise specifies, and its three self tests (N rate, soil type, residue and initial N).

**Data.** Course data in `notebooks/course-data/`: `PN_1976_Summary.xls` and `PN_1976_TimeSeriesData.xls`
(Hang et al. 1984). From [DSSAT/dssat-csm-data](https://github.com/DSSAT/dssat-csm-data) (BSD-3): peanut
UFGA7601 and UFGA7604, maize UFGA8201, weather and soils.

In [1]:
import sys, time, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))      # docs/notebooks
import learnlib as L
from sol import read_sol

M = "module4"
print("engine   :", L.config.engine_exe())
print("csm-data :", L.CSM_DATA, "@", L.CSM_DATA_COMMIT[:8])

engine   : C:\TEMP\dsn\dssat\dscsm048.exe
csm-data : C:\Users\benny\OneDrive\Documents\Github\dssat-csm-data @ 99c95b34


## 1. Creating crop measurement files

A FileA holds one line per treatment of what was measured at the end of the season; a FileT holds one line per
treatment and sampling date. The columns are DSSAT variable codes, defined in the engine's `DATA.CDE`, and the
files are fixed-format: 6 characters per column, right-aligned under the code.

### The shading study's data

The course gives the final harvest (FileA) and the growth analysis (FileT) of a control and a shaded treatment
as two spreadsheets. Their header row already holds the DSSAT codes:

In [2]:
summ_x = pd.read_excel(L.COURSE / "PN_1976_Summary.xls", header=0).rename(columns={"@TRNO": "TRNO"})
ts_x = pd.read_excel(L.COURSE / "PN_1976_TimeSeriesData.xls", header=0).rename(columns={"@TRNO": "TRNO"})
codes = L.data_codes([c for c in summ_x.columns if c != "TRNO"] + [c for c in ts_x.columns if c not in ("TRNO", "DATE")])
codes["file"] = ["FileA"] * (len(summ_x.columns) - 1) + ["FileT"] * (len(ts_x.columns) - 2)
display(summ_x, ts_x.head(6))
codes

,TRNO,PWAM,H#AM,HWAM,CWAM,HWUM
0,1,6718,942,5283,14240,0.5608
1,2,5037,658,3611,11431,0.5488


,TRNO,DATE,PWAD,G#AD,GWAD,P#AD,CWAD,GWGD
0,1,76161,10,0,0,0,1407,0.0
1,1,76168,28,0,0,0,2268,0.0
2,1,76175,99,0,0,11,3032,0.0
3,1,76182,362,0,0,69,4365,0.0
4,1,76189,727,85,99,192,6498,116.5
5,1,76196,1305,313,448,329,7426,143.1


,code,label,description,file
0,PWAM,Pod/Ear w kg/ha,Pod/Ear/Panicle weight at maturity (kg [dm]/ha),FileA
1,H#AM,Number #/m2,Number at maturity (no/m2),FileA
2,HWAM,Mat Yield kg/ha,Yield at harvest maturity (kg [dm]/ha),FileA
3,CWAM,Biomass kg/ha,Biomass at maturity (total crop minus roots) (...,FileA
4,HWUM,Weight g/unit,Unit wt at maturity (g [dm]/unit),FileA
5,PWAD,Pod wt kg/ha,Pod weight (kg [dm]/ha),FileT
6,G#AD,Grain no #/m2,Grain number (no/m2),FileT
7,GWAD,Grain wt kg/ha,Grain weight (kg [dm]/ha),FileT
8,P#AD,Pod no #/m2,Pod number (no/m2),FileT
9,CWAD,Biomass kg/ha,Total crop weight minus roots (kg [dm]/ha),FileT


### Building the experiment: `UFGA7609.PNX`

The exercise starts from the 1976 Florunner experiment `UFGA7601.PNX` and changes only what differs: the
experiment number (9), the planting and simulation start date (22 April 1976), automatic irrigation, and a
second treatment with an **environmental modification**: solar radiation set to 20% of normal for 21 days, from
4 to 24 August 1976, and back to normal on 25 August. The notebook makes the same edits to the text of the file,
section by section, as XBuild would.

In [3]:
src = (L.CSM_DATA / "Peanut/UFGA7601.PNX").read_text(encoding="latin-1").replace(chr(26), "")
x = src.replace("*EXP.DETAILS: UFGA7601PN 1976 IRRIGATED, FLORUNNER", "*EXP.DETAILS: UFGA7609PN 1976 SHADING STUDY, FLORUNNER")
x = L.replace_section(x, "TREATMENTS", '''*TREATMENTS                        -------------FACTOR LEVELS------------
@N R O C TNAME.................... CU FL SA IC MP MI MF MR MC MT ME MH SM
 1 1 0 0 CONTROL, NO SHADE          1  1  0  1  1  0  1  0  0  0  0  1  1
 2 1 0 0 SHADE 20%, 4-24 AUG        1  1  0  1  1  0  1  0  0  0  1  1  1''')
x = re.sub(r"(?m)^( 1    PN )76126", r"\g<1>76113", x)                 # initial conditions measured on the start date
x = re.sub(r"(?m)^( 1 )76126(   -99  12.9)", r"\g<1>76113\g<2>", x)       # planting date
x = re.sub(r"(?m)^( 1 GE              1     1     S )76126", r"\g<1>76113", x)   # start of simulation
x = L.set_switches(x, {"IRRIG": "A"})                                    # automatic irrigation
ENV = '''*ENVIRONMENT MODIFICATIONS
@E ODATE EDAY  ERAD  EMAX  EMIN  ERAIN ECO2  EDEW  EWIND ENVNAME
 1 76217 A   0 M .20 A   0 A   0 A 0.0 A   0 A   0 A   0
 1 76238 A   0 M   1 A   0 A   0 A 0.0 A   0 A   0 A   0
'''
x = x.replace("*HARVEST DETAILS", ENV + "\n*HARVEST DETAILS")
assert x.count("76113") >= 3, "dates not replaced"
d1 = L.work_dir("m4_shade")
L.stage(d1, "Weather/UFGA7601.WTH", "Soil/SOIL.SOL")
(d1 / "UFGA7609.PNX").write_text(x, encoding="latin-1")
print(x[x.index("*TREATMENTS"):x.index("*CULTIVARS")])
print(x[x.index("*ENVIRONMENT"):x.index("*HARVEST")])

*TREATMENTS                        -------------FACTOR LEVELS------------
@N R O C TNAME.................... CU FL SA IC MP MI MF MR MC MT ME MH SM
 1 1 0 0 CONTROL, NO SHADE          1  1  0  1  1  0  1  0  0  0  0  1  1
 2 1 0 0 SHADE 20%, 4-24 AUG        1  1  0  1  1  0  1  0  0  0  1  1  1


*ENVIRONMENT MODIFICATIONS
@E ODATE EDAY  ERAD  EMAX  EMIN  ERAIN ECO2  EDEW  EWIND ENVNAME
 1 76217 A   0 M .20 A   0 A   0 A 0.0 A   0 A   0 A   0
 1 76238 A   0 M   1 A   0 A   0 A 0.0 A   0 A   0 A   0




In the `ERAD` column, `M .20` multiplies radiation by 0.20 from 4 August (day 217); `M   1` restores it on
25 August (day 238). A modification stays in force until another one replaces it, which is why the second row
is needed.

### Writing the FileA and FileT

`learnlib.write_observed` writes both files from the spreadsheets. The engine reads them by column position,
so the alignment has to be exact: a first version of this writer put every name one character to the left of
its values; read back by whitespace the file looked fine, but the engine's evaluation reported anthesis about
11,000 days after planting. Comparing the engine's `Evaluate` output with the spreadsheet is the check.

In [4]:
ts_x["DATE"] = [L._date(v) for v in ts_x.DATE]
L.write_observed(d1 / "UFGA7609.PNA", summ_x, "A", "UFGA7609PN", "Environmental Modification")
L.write_observed(d1 / "UFGA7609.PNT", ts_x, "T", "UFGA7609PN", "Environmental Modification")
print((d1 / "UFGA7609.PNA").read_text()); print("\n".join((d1 / "UFGA7609.PNT").read_text().splitlines()[:6]))
s1, dd1, _ = L.run(d1, "UFGA7609.PNX")
ev = dd1["evaluate"].apply(pd.to_numeric, errors="coerce")
pairs = []
for _, r in ev.iterrows():
    for c in ev.columns:
        if c.endswith("S") and c[:-1] + "M" in ev.columns:
            sv, mv = r[c], r[c[:-1] + "M"]
            if not (np.isnan(sv) or np.isnan(mv)):
                pairs.append({"TRNO": int(r.TN), "code": c[:-1], "simulated": sv, "measured": mv})
shade_eval = pd.DataFrame(pairs).merge(L.data_codes(sorted({p["code"] for p in pairs}))[["code", "description"]], on="code", how="left")
shade_eval["sim/obs"] = shade_eval.simulated / shade_eval.measured
shade_eval

*EXP.DATA (A): UFGA7609PN Environmental Modification

@TRNO   PWAM  H#AM  HWAM  CWAM  HWUM
     1  6718   942  5283 14240 .5608
     2  5037   658  3611 11431 .5488

*EXP.DATA (T): UFGA7609PN Environmental Modification

@TRNO   DATE  PWAD  G#AD  GWAD  P#AD  CWAD  GWGD
     1 76161    10     0     0     0  1407     0
     1 76168    28     0     0     0  2268     0
     1 76175    99     0     0    11  3032     0


,TRNO,code,simulated,measured,description,sim/obs
0,1,HWAM,5115.0000,5283.0000,Yield at harvest maturity (kg [dm]/ha),0.968200
1,1,PWAM,6438.0000,6718.0000,Pod/Ear/Panicle weight at maturity (kg [dm]/ha),0.958321
2,1,CWAM,14118.0000,14240.0000,Biomass at maturity (total crop minus roots) (...,0.991433
3,1,H#AM,873.0000,942.0000,Number at maturity (no/m2),0.926752
4,1,HWUM,0.5859,0.5608,Unit wt at maturity (g [dm]/unit),1.044757
5,2,HWAM,4170.0000,3611.0000,Yield at harvest maturity (kg [dm]/ha),1.154805
6,2,PWAM,5266.0000,5037.0000,Pod/Ear/Panicle weight at maturity (kg [dm]/ha),1.045464
7,2,CWAM,12017.0000,11431.0000,Biomass at maturity (total crop minus roots) (...,1.051264
8,2,H#AM,775.0000,658.0000,Number at maturity (no/m2),1.177812
9,2,HWUM,0.5381,0.5488,Unit wt at maturity (g [dm]/unit),0.980503


In [5]:
pg = dd1["plantgro"]
pdate_p = pd.Timestamp("1976-04-22")
tvars = [c for c in ["PWAD", "G#AD", "GWAD", "P#AD", "CWAD", "GWGD"] if c in pg.columns]
shade_sim = pg[["TRNO", "date", "DAP"] + tvars]
shade_obs = ts_x[["TRNO", "DATE"] + tvars].rename(columns={"DATE": "date"})
shade_obs["DAP"] = (shade_obs.date - pdate_p).dt.days
st = []
for (t, v), o in shade_obs.melt(id_vars=["TRNO", "date", "DAP"], var_name="variable", value_name="obs").dropna().groupby(["TRNO", "variable"]):
    sim = shade_sim[shade_sim.TRNO == t].set_index("date")
    o = o[o.date.isin(sim.index)]
    sv = sim.loc[o.date, v].astype(float)
    st.append({"TRNO": t, "variable": v, "n": len(o), "rmse": L.rmse(o.obs, sv), "d": L.willmott_d(o.obs, sv)})
shade_stats = pd.DataFrame(st)
shade_stats

,TRNO,variable,n,rmse,d
0,1,CWAD,13,1168.345543,0.979878
1,1,G#AD,13,146.599664,0.963029
2,1,GWAD,13,447.629139,0.986172
3,1,GWGD,13,29.257332,0.994307
4,1,P#AD,13,127.975659,0.928809
5,1,PWAD,13,764.970537,0.975743
6,2,CWAD,4,506.247469,0.889125
7,2,G#AD,4,89.262254,0.390711
8,2,GWAD,4,503.647694,0.792966
9,2,GWGD,4,81.832422,0.757863


### The whole study, and when shade hurts most

The published experiment `UFGA7604.PNX` of dssat-csm-data has the full study: a control and four 21-day shade
periods at 25% of normal radiation, from early in the season to pod filling. And with the shade exercise's
experiment as a template, the notebook moves a 21-day window through the season in steps of a week, at two
strengths, in one FileX with one environmental-modification level per treatment.

In [6]:
d2 = L.work_dir("m4_shade_study")
L.stage(d2, "Peanut/UFGA7604.PNX", "Peanut/UFGA7604.PNA", "Weather/UFGA7601.WTH", "Soil/SOIL.SOL")
s2, _, _ = L.run(d2, "UFGA7604.PNX", outputs="summary")
obs04 = L._read_table(d2 / "UFGA7604.PNA")
study = s2[["TRNO", "TNAM", "HWAM", "CWAM", "H#AM", "HWUM"]].merge(obs04[["TRNO", "HWAM", "CWAM", "H#AM"]].add_suffix("_obs").rename(columns={"TRNO_obs": "TRNO"}), on="TRNO", how="left")
study["TNAM"] = study.TNAM.str.strip()

starts = list(range(154, 239, 7))
strengths = [0.25, 0.50]
trt = [" 1 1 0 0 NO SHADE                   1  1  0  1  1  0  1  0  0  0  0  1  1"]
env = ["*ENVIRONMENT MODIFICATIONS", "@E ODATE EDAY  ERAD  EMAX  EMIN  ERAIN ECO2  EDEW  EWIND ENVNAME"]
meta = [{"TRNO": 1, "start": np.nan, "radiation": 1.0}]
lev = 0
for f in strengths:
    for st_ in starts:
        lev += 1
        n = lev + 1
        name = f"SHADE {int(f * 100)}% FROM DAY {st_}"
        trt.append(f"{n:>2} 1 0 0 {name:<25}{1:>3}{1:>3}{0:>3}{1:>3}{1:>3}{0:>3}{1:>3}{0:>3}{0:>3}{0:>3}{lev:>3}{1:>3}{1:>3}")
        env.append(f"{lev:>2} 76{st_:03d} A   0 M{('%.2f' % f)[1:]:>4} A   0 A   0 A 0.0 A   0 A   0 A   0")
        env.append(f"{lev:>2} 76{st_ + 21:03d} A   0 M   1 A   0 A   0 A 0.0 A   0 A   0 A   0")
        meta.append({"TRNO": n, "start": st_, "radiation": f})
xs = L.replace_section(x, "TREATMENTS", "*TREATMENTS                        -------------FACTOR LEVELS------------\n"
                       "@N R O C TNAME.................... CU FL SA IC MP MI MF MR MC MT ME MH SM\n" + "\n".join(trt))
xs = L.replace_section(xs, "ENVIRONMENT MODIFICATIONS", "\n".join(env))
print("\n".join(env[:4]))
d3 = L.work_dir("m4_shade_sweep")
L.stage(d3, "Weather/UFGA7601.WTH", "Soil/SOIL.SOL")
(d3 / "UFGA7608.PNX").write_text(xs, encoding="latin-1")
s3, dd3, _ = L.run(d3, "UFGA7608.PNX")
sweep = s3[["TRNO", "HWAM", "CWAM", "H#AM", "HWUM"]].merge(pd.DataFrame(meta), on="TRNO")
for c in ["HWAM", "CWAM", "H#AM", "HWUM"]:
    sweep[c] = pd.to_numeric(sweep[c], errors="coerce")
sweep["start_dap"] = sweep.start - 113
sweep["loss"] = 1 - sweep.HWAM / float(sweep.loc[sweep.TRNO == 1, "HWAM"].iloc[0])
sweep_growth = dd3["plantgro"][["TRNO", "DAP", "GWAD", "CWAD", "LAID"]]
sweep_growth = sweep_growth[sweep_growth.DAP >= 0]
display(study)
sweep.pivot(index="start", columns="radiation", values="HWAM")

*ENVIRONMENT MODIFICATIONS
@E ODATE EDAY  ERAD  EMAX  EMIN  ERAIN ECO2  EDEW  EWIND ENVNAME
 1 76154 A   0 M .25 A   0 A   0 A 0.0 A   0 A   0 A   0
 1 76175 A   0 M   1 A   0 A   0 A 0.0 A   0 A   0 A   0


,TRNO,TNAM,HWAM,CWAM,H#AM,HWUM,HWAM_obs,CWAM_obs,H#AM_obs
0,1,CONTROL- NO SHADE,5034,13162,871,0.577839,5283,14240,942
1,2,21D SHADE- 154-174,4143,11742,842,0.492103,5344,13724,967
2,3,21D SHADE- 175-195,2845,11963,653,0.435442,4511,13143,795
3,4,21D SHADE- 196-216,2684,12245,592,0.453101,3597,10786,720
4,5,21D SHADE- 217-237,3975,11211,780,0.509775,3611,11431,658


radiation,0.25,0.50,1.00
start,,,
NaN,NaN,NaN,5115.0
154.0,4527.0,4930.0,NaN
161.0,4293.0,4855.0,NaN
168.0,3875.0,4573.0,NaN
175.0,3273.0,4225.0,NaN
182.0,2540.0,3979.0,NaN
189.0,2385.0,3557.0,NaN
196.0,2794.0,3418.0,NaN
203.0,3367.0,3612.0,NaN


In [7]:
L.export(codes, M, "codes"); L.export(summ_x, M, "shade_filea"); L.export(ts_x, M, "shade_filet")
L.export(shade_eval, M, "shade_evaluate"); L.export(shade_sim, M, "shade_sim"); L.export(shade_obs, M, "shade_obs")
L.export(shade_stats, M, "shade_stats"); L.export(study, M, "shade_study"); L.export(sweep, M, "shade_sweep")
L.export(sweep_growth, M, "shade_sweep_growth")
L.export(pd.DataFrame([{"section": "TREATMENTS", "text": x[x.index("*TREATMENTS"):x.index("*CULTIVARS")].rstrip()},
                       {"section": "ENVIRONMENT MODIFICATIONS", "text": x[x.index("*ENVIRONMENT"):x.index("*HARVEST")].rstrip()},
                       {"section": "FileA", "text": (d1 / "UFGA7609.PNA").read_text().rstrip()},
                       {"section": "FileT (first lines)", "text": "\n".join((d1 / "UFGA7609.PNT").read_text().splitlines()[:8])}]), M, "shade_files")

WindowsPath('C:/Users/benny/OneDrive/Documents/Github/xdssat/docs/learn/data/module4/shade_files.csv')

## 2. Nitrogen-limited production: `UFGA8213.MZX`

The exercise builds on the water-limited maize experiment of module 3 and adds what the nitrogen balance needs:

- initial conditions measured on 25 February 1982: previous crop maize, 100 kg/ha of roots, no nodules or
  rhizobia, 1000 kg/ha of crop residue with 0.8% N, fully incorporated to 15 cm;
- the initial ammonium and nitrate of each soil layer (1.5 g/Mg of NH4 in the top four layers, 0.6 in the next
  two and 0.5 in the last two; 0.6 g/Mg of NO3 throughout), with the same soil water as before;
- water **and** nitrogen simulated; nitrogen fertilizer on reported dates;
- four treatments: rainfed and automatically irrigated (30 cm, 50%), each without N and with 60 kg N/ha on
  15 March plus 75 kg N/ha on 12 April as ammonium nitrate, broadcast and incorporated at 10 cm.

The soil N model keeps ammonium and nitrate in every layer; they change with fertilizer, mineralization,
nitrification, leaching and uptake ([`Soil/Inorganic_N/SOILNI.for`](https://github.com/DSSAT/dssat-csm-os/blob/25c93b9a2f73f8ec45bf441b198dac5fb7b22849/Soil/Inorganic_N/SOILNI.for)). In the
plant, CERES-Maize compares the N concentration of the tops with a critical (`TCNP`) and a minimum (`TMNC`)
concentration: `NFAC = 1 - (TCNP - TANC) / (TCNP - TMNC)`
([`MZ_NFACTO.for:61`](https://github.com/DSSAT/dssat-csm-os/blob/25c93b9a2f73f8ec45bf441b198dac5fb7b22849/Plant/CERES-Maize/MZ_NFACTO.for#L61)).

In [8]:
base = (L.CSM_DATA / "Maize/UFGA8201.MZX").read_text(encoding="latin-1").replace(chr(26), "")   # drop the DOS end-of-file mark
NH4 = [1.5, 1.5, 1.5, 1.5, 0.6, 0.6, 0.5, 0.5]
H2O = [0.086, 0.086, 0.086, 0.086, 0.076, 0.076, 0.130, 0.258]
BOT = [5, 15, 30, 60, 90, 120, 150, 180]

def n_filex(treatments, fert_levels):
    t = base.replace(base.splitlines()[0], "*EXP.DETAILS: UFGA8213MZ NITROGEN BALANCE SIMULATION")
    t = L.replace_section(t, "TREATMENTS", "*TREATMENTS                        -------------FACTOR LEVELS------------\n"
                          "@N R O C TNAME.................... CU FL SA IC MP MI MF MR MC MT ME MH SM\n" + "\n".join(
                              f"{i:>2} 1 0 0 {nm:<25}{1:>3}{1:>3}{0:>3}{1:>3}{1:>3}{0:>3}{mf:>3}{0:>3}{0:>3}{0:>3}{0:>3}{0:>3}{sm:>3}"
                              for i, (nm, mf, sm) in enumerate(treatments, 1)))
    t = L.replace_section(t, "INITIAL CONDITIONS", "*INITIAL CONDITIONS\n"
                          "@C   PCR ICDAT  ICRT  ICND  ICRN  ICRE  ICWD ICRES ICREN ICREP ICRIP ICRID ICNAME\n"
                          " 1    MZ 82056   100     0     0     0   -99  1000    .8     0   100    15 -99\n"
                          "@C  ICBL  SH2O  SNH4  SNO3\n" + "\n".join(f" 1 {b:>5} {w:>5.3f} {n:>5} {0.6:>5}" for b, w, n in zip(BOT, H2O, NH4)))
    rows = [f"{lvl:>2} {date} FE001 AP002    10 {amt:>5}     0     0     0     0   -99 -99"
            for lvl, events in enumerate(fert_levels, 1) for date, amt in events]
    t = L.replace_section(t, "FERTILIZERS (INORGANIC)", "*FERTILIZERS (INORGANIC)\n"
                          "@F FDATE  FMCD  FACD  FDEP  FAMN  FAMP  FAMK  FAMC  FAMO  FOCD FERNAME\n" + "\n".join(rows))
    sim = t[t.index("*SIMULATION CONTROLS"):]
    blk = [ln for ln in sim.splitlines()[1:] if ln.strip()]
    lvl2 = [(" 2" + ln[2:] if ln[:2] == " 1" else ln) for ln in blk]
    t = t[:t.index("*SIMULATION CONTROLS")] + "*SIMULATION CONTROLS\n" + "\n".join(blk) + "\n\n" + "\n".join(lvl2) + "\n"
    t = L.set_switches(t, {"WATER": "Y", "NITRO": "Y", "IRRIG": "R"})
    lines = t.splitlines()
    for i, ln in enumerate(lines):                  # level 2: irrigate automatically
        if ln.startswith(" 2 MA "):
            hdr = next(h for h in reversed(lines[:i]) if h.startswith("@N MANAGEMENT"))
            lines[i] = L.set_switches("*SIMULATION CONTROLS\n" + hdr + "\n" + ln, {"IRRIG": "A"}).splitlines()[2]
    return "\n".join(lines) + "\n"

TRT = [("RAINFED, NO N", 0, 1), ("AUTO IRRIGATED, NO N", 0, 2), ("RAINFED, 135 KG N", 1, 1), ("AUTO IRRIGATED, 135 KG N", 1, 2)]
x13 = n_filex(TRT, [[(82074, 60), (82102, 75)]])
d = L.work_dir("m4_n")
L.stage(d, "Weather/UFGA8201.WTH", "Soil/SOIL.SOL")
(d / "UFGA8213.MZX").write_text(x13, encoding="latin-1")
for sec in ("*TREATMENTS", "*INITIAL CONDITIONS", "*PLANTING DETAILS", "*FERTILIZERS"):
    i = x13.index(sec); print(x13[i:x13.index("*", i + 1)])
print("\n".join(ln for ln in x13.splitlines() if ln[3:6] in ("MA ", "OP ")))
sN, dN, _ = L.run(d, "UFGA8213.MZX")
nbal = sN[["TRNO", "HWAM", "CWAM", "NICM", "NUCM", "NLCM", "NIAM", "NMINC", "CNAM", "GNAM", "PRCM", "IRCM", "DRCM"]].apply(pd.to_numeric, errors="coerce")
nbal.insert(1, "treatment", nbal.TRNO.map({1: "rainfed, no N", 2: "irrigated, no N", 3: "rainfed, 135 kg N", 4: "irrigated, 135 kg N"}))
nbal

*TREATMENTS                        -------------FACTOR LEVELS------------
@N R O C TNAME.................... CU FL SA IC MP MI MF MR MC MT ME MH SM
 1 1 0 0 RAINFED, NO N              1  1  0  1  1  0  0  0  0  0  0  0  1
 2 1 0 0 AUTO IRRIGATED, NO N       1  1  0  1  1  0  0  0  0  0  0  0  2
 3 1 0 0 RAINFED, 135 KG N          1  1  0  1  1  0  1  0  0  0  0  0  1
 4 1 0 0 AUTO IRRIGATED, 135 KG N   1  1  0  1  1  0  1  0  0  0  0  0  2


*INITIAL CONDITIONS
@C   PCR ICDAT  ICRT  ICND  ICRN  ICRE  ICWD ICRES ICREN ICREP ICRIP ICRID ICNAME
 1    MZ 82056   100     0     0     0   -99  1000    .8     0   100    15 -99
@C  ICBL  SH2O  SNH4  SNO3
 1     5 0.086   1.5   0.6
 1    15 0.086   1.5   0.6
 1    30 0.086   1.5   0.6
 1    60 0.086   1.5   0.6
 1    90 0.076   0.6   0.6
 1   120 0.076   0.6   0.6
 1   150 0.130   0.5   0.6
 1   180 0.258   0.5   0.6


*PLANTING DETAILS
@P PDATE EDATE  PPOP  PPOE  PLME  PLDS  PLRS  PLRD  PLDP  PLWT  PAGE  PENV  PLPH  SPRL                        

,TRNO,treatment,HWAM,CWAM,NICM,NUCM,NLCM,NIAM,NMINC,CNAM,GNAM,PRCM,IRCM,DRCM
0,1,"rainfed, no N",547,2505,0,25,44,13,47,21,6,661,0,411
1,2,"irrigated, no N",882,2992,0,27,46,13,50,23,9,661,74,451
2,3,"rainfed, 135 kg N",2268,7086,135,114,66,34,44,85,39,661,0,321
3,4,"irrigated, 135 kg N",8436,16373,135,130,70,21,51,110,80,661,200,370


In [9]:
pdate_m = pd.Timestamp("1982-02-26")
sn = dN["soilni"]
ni_cols = [c for c in sn.columns if re.fullmatch(r"NI\d+D", c)]
soil_n = sn[["TRNO", "date", "NIAD", "NITD", "NHTD", "NLCC", "NMNC", "NAPC", "NUCM"]].copy()
soil_n["DAP"] = (soil_n.date - pdate_m).dt.days
no3_long = sn[["TRNO", "date"] + ni_cols].melt(id_vars=["TRNO", "date"], var_name="layer", value_name="NO3")
no3_long["layer"] = no3_long.layer.str.extract(r"(\d+)").astype(int)
no3_long["DAP"] = (no3_long.date - pdate_m).dt.days
mill = read_sol(L.CSM_DATA / "Soil/SOIL.SOL", ids=["IBMZ910014"])["IBMZ910014"].layers
bottoms = L.engine_layers(mill.SLB.tolist())
layers = pd.DataFrame({"layer": range(1, len(bottoms) + 1), "top": [0.0] + bottoms[:-1], "bottom": bottoms})
pg = dN["plantgro"][["TRNO", "date", "DAP", "NSTD", "WSGD", "CWAD", "GWAD", "LAID"]]
pn = dN["plantn"]
pcols = [c for c in ["CNAD", "GNAD", "VNAD", "GN%D", "VN%D", "NUPC"] if c in pn.columns]
plant_n = pg.merge(pn[["TRNO", "date"] + pcols], on=["TRNO", "date"], how="left")
plant_n = plant_n[plant_n.DAP >= 0]
L.export(nbal, M, "n_balance"); L.export(soil_n, M, "soil_n"); L.export(no3_long.drop(columns="date"), M, "soil_no3_layers")
L.export(layers, M, "engine_layers"); L.export(plant_n, M, "plant_n")
L.export(pd.DataFrame([{"section": s_, "text": x13[x13.index(s_):x13.index("*", x13.index(s_) + 1)].rstrip()}
                       for s_ in ("*TREATMENTS", "*INITIAL CONDITIONS", "*FERTILIZERS")]), M, "n_files")
plant_n.groupby("TRNO")[["NSTD"] + pcols].max().round(2)

,NSTD,CNAD,GNAD,VNAD,GN%D,VN%D,NUPC
TRNO,,,,,,,
1,0.60,20.69,6.41,15.29,1.05,4.4,24.95
2,0.58,23.12,9.66,15.81,1.01,4.4,27.18
3,0.05,92.48,42.56,92.48,1.74,4.4,114.08
4,0.60,109.80,79.61,102.00,1.27,4.4,129.81


### Two soil organic matter models

DSSAT has two ways to simulate soil organic matter, chosen with the `MESOM` switch of the simulation controls:
`G`, the original CERES approach (Godwin), and `P`, the CENTURY-based model (Parton). The four treatments of the
exercise under both:

In [10]:
som = []
for opt, name in [("G", "CERES (Godwin)"), ("P", "CENTURY (Parton)")]:
    d = L.work_dir(f"m4_som_{opt}")
    L.stage(d, "Weather/UFGA8201.WTH", "Soil/SOIL.SOL")
    (d / "UFGA8213.MZX").write_text(L.set_switches(x13, {"MESOM": opt}), encoding="latin-1")
    s, _, _ = L.run(d, "UFGA8213.MZX", outputs="summary")
    som.append(pd.DataFrame({"model": name, "treatment": s.TRNO.map(nbal.set_index("TRNO").treatment),
                             **{c: pd.to_numeric(s[c], errors="coerce") for c in ["HWAM", "NUCM", "NLCM", "NMINC", "NIAM"]}}))
som = pd.concat(som)
L.export(som, M, "n_som")
som.pivot(index="treatment", columns="model", values=["HWAM", "NMINC"])

HWAM                           NMINC  \
model               CENTURY (Parton) CERES (Godwin) CENTURY (Parton)   
treatment                                                              
irrigated, 135 kg N             8097           8436               33   
irrigated, no N                  368            882               29   
rainfed, 135 kg N               2600           2268               24   
rainfed, no N                    250            547               27   

                                    
model               CERES (Godwin)  
treatment                           
irrigated, 135 kg N             51  
irrigated, no N                 50  
rainfed, 135 kg N               44  
rainfed, no N                   47

### Self test 1: the response to the N rate

0, 50, 100, 150 and 200 kg N/ha in total, in the two applications of the exercise (in the same 60:75
proportion), rainfed and automatically irrigated.

In [11]:
rates = [0, 50, 100, 150, 200]
fl = [[(82074, round(r * 60 / 135)), (82102, r - round(r * 60 / 135))] for r in rates if r > 0]
trts = [(f"{w} {r} KG N", 0 if r == 0 else rates.index(r), sm) for sm, w in [(1, "RAINFED"), (2, "AUTO IRRIGATED")] for r in rates]
xr = n_filex(trts, fl)
d = L.work_dir("m4_nrates")
L.stage(d, "Weather/UFGA8201.WTH", "Soil/SOIL.SOL")
(d / "UFGA8213.MZX").write_text(xr, encoding="latin-1")
s, dd, _ = L.run(d, "UFGA8213.MZX")
nresp = s[["TRNO", "HWAM", "CWAM", "NICM", "NUCM", "NLCM", "CNAM", "GNAM"]].apply(pd.to_numeric, errors="coerce")
nresp["water"] = ["rainfed" if t <= len(rates) else "irrigated" for t in nresp.TRNO]
nresp["rate"] = [rates[(t - 1) % len(rates)] for t in nresp.TRNO]
zero = nresp[nresp.rate == 0].set_index("water")
nresp["agronomic_efficiency"] = (nresp.HWAM - nresp.water.map(zero.HWAM)) / nresp.rate.replace(0, np.nan)
nresp["recovery"] = (nresp.NUCM - nresp.water.map(zero.NUCM)) / nresp.rate.replace(0, np.nan)
nstress = dd["plantgro"][["TRNO", "DAP", "NSTD"]].merge(nresp[["TRNO", "water", "rate"]], on="TRNO").drop(columns="TRNO")
nstress = nstress[nstress.DAP >= 0]
L.export(nresp, M, "n_response"); L.export(nstress, M, "n_response_stress")
nresp.pivot(index="rate", columns="water", values="HWAM")

water,irrigated,rainfed
rate,,
0,882,547
50,3195,1374
100,6205,2309
150,9140,2161
200,10389,1530


### Self test 2: three soils

The four treatments on a clay, a silt loam and a sand from the generic profiles of the soil file (the deep ones,
210 cm), next to the Millhopper fine sand. For those three soils the initial soil water is rebuilt at 100% of
their plant-available range (`ic_water=1.0`); ammonium and nitrate keep the exercise's values by depth.

In [12]:
SOILS = {"IB00000001": "deep silty clay", "IB00000004": "deep silt loam", "IB00000010": "deep sand", "IBMZ910014": "Millhopper fine sand"}
profiles = read_sol(L.CSM_DATA / "Soil/SOIL.SOL", ids=list(SOILS))
dtp = L.work_dir("m4_soils")
L.stage(dtp, "Weather/UFGA8201.WTH", "Soil/SOIL.SOL")
(dtp / "TEMPLATE.MZX").write_text(x13, encoding="latin-1")
fx13 = L.filex.FileX.read(dtp / "TEMPLATE.MZX")
u = pd.DataFrame([{"cell": f"{sid}_{t}", "name": f"{SOILS[sid][:14]} T{t}", "wsta": "UFGA", "soil_id": sid, "pdate": pdate_m,
                   "trno_template": t} for sid in SOILS for t in (1, 2, 3, 4)])
others = {k: v for k, v in profiles.items() if k != "IBMZ910014"}
built = L.filex.build_filex(fx13, u, exp_code="UFGA8214", title="N balance on four soils", soils=others, ic_water=1.0)
(dtp / "UFGA8214.MZX").write_text(built.text, encoding="latin-1")
s, _, _ = L.run(dtp, "UFGA8214.MZX", outputs="summary")
n_soils = s.merge(built.units[["TRNO", "soil_id", "trno_template"]], on="TRNO")
n_soils = pd.DataFrame({"soil": n_soils.soil_id.map(SOILS), "treatment": n_soils.trno_template.map(nbal.set_index("TRNO").treatment),
                        **{c: pd.to_numeric(n_soils[c], errors="coerce") for c in ["HWAM", "NUCM", "NLCM", "NIAM", "DRCM"]}})
L.export(n_soils, M, "n_soils")
n_soils.pivot(index="soil", columns="treatment", values="HWAM")

treatment,"irrigated, 135 kg N","irrigated, no N","rainfed, 135 kg N","rainfed, no N"
soil,,,,
Millhopper fine sand,8436,882,2268,547
deep sand,8392,351,4773,301
deep silt loam,11875,2837,4466,2091
deep silty clay,12752,5599,11352,5161


### Self test 3: residue and initial nitrogen

The fertilized treatments (3 and 4) with the crop residue, its N content and the initial nitrate changed one at a
time, through the initial-conditions columns `ICRES`, `ICREN` and `SNO3` (the nitrate value is then the same in
every layer).

In [13]:
SCEN = [("as in the exercise", {}), ("no residue", {"IC.ICRES": 0}), ("3000 kg/ha residue", {"IC.ICRES": 3000}),
        ("residue 0.4% N", {"IC.ICREN": 0.4}), ("residue 1.6% N", {"IC.ICREN": 1.6}),
        ("NO3 3 g/Mg in every layer", {"IC.SNO3": 3}), ("NO3 10 g/Mg in every layer", {"IC.SNO3": 10})]
rows = []
for name, ov in SCEN:
    for t in (3, 4):
        rows.append({"cell": f"{name}_{t}", "name": f"{name[:20]} T{t}", "wsta": "UFGA", "soil_id": "IBMZ910014", "pdate": pdate_m,
                     "trno_template": t, "scenario": name, **ov})
u = pd.DataFrame(rows)
dres = L.work_dir("m4_residue")
L.stage(dres, "Weather/UFGA8201.WTH", "Soil/SOIL.SOL")
built = L.filex.build_filex(fx13, u.drop(columns="scenario"), exp_code="UFGA8215", title="Residue and initial N")
(dres / "UFGA8215.MZX").write_text(built.text, encoding="latin-1")
s, _, _ = L.run(dres, "UFGA8215.MZX", outputs="summary")
s = s.merge(built.units[["TRNO", "cell", "trno_template"]], on="TRNO").merge(u[["cell", "scenario"]], on="cell")
n_res = pd.DataFrame({"scenario": s.scenario, "treatment": s.trno_template.map(nbal.set_index("TRNO").treatment),
                      **{c: pd.to_numeric(s[c], errors="coerce") for c in ["HWAM", "NUCM", "NLCM", "NIAM", "NMINC"]}})
L.export(n_res, M, "n_residue")
n_res.pivot(index="scenario", columns="treatment", values="HWAM")

treatment,"irrigated, 135 kg N","rainfed, 135 kg N"
scenario,,
3000 kg/ha residue,8821,2303
NO3 10 g/Mg in every layer,8541,2290
NO3 3 g/Mg in every layer,8462,2306
as in the exercise,8436,2268
no residue,8233,2310
residue 0.4% N,8270,2289
residue 1.6% N,8684,2326


### Nitrogen fixation in a legume

Grain legumes add a source of nitrogen that cereals lack: symbiotic fixation by root nodules. CROPGRO grows
nodules when the crop's nitrogen demand exceeds what roots take up from the soil and mobilize from old tissue,
and the nodules fix nitrogen at a rate set by their mass and by temperature, the crop's water status, flooding,
nodule age and the carbohydrate they receive (`Plant/CROPGRO/NFIX.for`). The `SYMBI` switch in the simulation controls turns fixation on or off.
Bragg soybean at Gainesville in 1978 (`UFGA7801.SBX`, a sandy soil without fertilizer), with the nitrogen balance
on and fixation on or off, and with the nitrogen balance off for comparison:

In [14]:
fx_sb = (L.CSM_DATA / "Soybean/UFGA7801.SBX").read_text(encoding="latin-1")
cases = {"no fixation": {"NITRO": "Y", "SYMBI": "N"}, "fixation": {"NITRO": "Y", "SYMBI": "Y"},
         "nitrogen not limiting": {"NITRO": "N", "SYMBI": "N"}}
trt_names = {1: "irrigated", 2: "rainfed"}
nfix, nfix_daily = [], []
for case, sw in cases.items():
    d = L.work_dir("m4_nfix_" + case.split()[0])
    L.stage(d, "Soil/SOIL.SOL", *[f"Weather/{w.name}" for w in (L.CSM_DATA / "Weather").glob("UFGA78*.WTH")])
    (d / "UFGA7801.SBX").write_text(L.set_switches(fx_sb, sw), encoding="latin-1")
    s, dd, _ = L.run(d, "UFGA7801.SBX")
    nfix.append(pd.DataFrame({"case": case, "treatment": s.TRNO.astype(int).map(trt_names),
                              **{c: pd.to_numeric(s[c], errors="coerce") for c in ["HWAM", "CWAM", "NUCM", "NFXM", "NLCM"]}}))
    if "plantn" in dd:
        pn = dd["plantn"][["TRNO", "DAP", "NFXC", "NUPC", "CNAD"]].copy()
        nfix_daily.append(pn.assign(case=case, treatment=pn.TRNO.astype(int).map(trt_names)).drop(columns="TRNO"))
nfix = pd.concat(nfix, ignore_index=True)
nfix_daily = pd.concat(nfix_daily, ignore_index=True)
L.export(nfix, M, "n_fixation"); L.export(nfix_daily, M, "n_fixation_daily", 1)
nfix

,case,treatment,HWAM,CWAM,NUCM,NFXM,NLCM
0,no fixation,irrigated,75,119,9,0,39.0
1,no fixation,rainfed,85,146,10,0,36.0
2,fixation,irrigated,2806,5988,14,276,34.0
3,fixation,rainfed,1088,3363,10,180,33.0
4,nitrogen not limiting,irrigated,2849,6016,0,0,NaN
5,nitrogen not limiting,rainfed,1115,3400,0,0,NaN


## 3. Things to try

- Write your own FileT for a field you know: one row per sampling date, the codes from `DATA.CDE`.
- In the shade sweep, find the 21 days when shade costs most. What is the crop doing then?
- Move all the nitrogen of treatment 4 to planting. What happens to leaching and to yield?

In [15]:
info = pd.DataFrame([{"engine": str(L.config.engine_exe()), "csm_data_commit": L.CSM_DATA_COMMIT,
                      "csm_os_commit": L.CSM_OS_COMMIT, "run_on": time.strftime("%Y-%m-%d %H:%M")}])
L.export(info, M, "run_info")
info.T

,0
engine,C:\TEMP\dsn\dssat\dscsm048.exe
csm_data_commit,99c95b340acb9515c2d0a8ab582f5fbbf076f345
csm_os_commit,25c93b9a2f73f8ec45bf441b198dac5fb7b22849
run_on,2026-11-07 10:53
